In [ ]:
import torch
from transformers import (
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    AutoConfig
)
from huggingface_hub import HfApi, login

login()

In [ ]:
import os
import json
import torch
from transformers import (
    AutoConfig,
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    AutoModel
)
from huggingface_hub import hf_hub_download, upload_file, HfApi
def download_existing_model():
    model_name = "cogniplayapp/wav2vec2-large-xls-r-300m-dm32"

    config_path = hf_hub_download(
        repo_id=model_name,
        filename="config.json",
        cache_dir="./temp_model"
    )

    model_path = hf_hub_download(
        repo_id=model_name,
        filename="pytorch_model.bin",
        cache_dir="./temp_model"
    )

    try:
        preprocessor_path = hf_hub_download(
            repo_id=model_name,
            filename="preprocessor_config.json",
            cache_dir="./temp_model"
        )
    except:
        print("No preprocessor config found, will create one")
        preprocessor_path = None

    return config_path, model_path, preprocessor_path
def fix_model_config(config_path):
    with open(config_path, 'r') as f:
        config_dict = json.load(f)

    print("Original config keys:", list(config_dict.keys()))
    config_dict.update({
        "architectures": ["Wav2Vec2ForSequenceClassification"],
        "model_type": "wav2vec2",
        "task_specific_params": {
            "audio-classification": {
                "problem_type": "single_label_classification",
                "num_labels": config_dict.get("num_labels", 2)
            }
        },
        "use_return_dict": True,
        "output_hidden_states": False,
        "output_attentions": False,
        "problem_type": "single_label_classification"
    })
    custom_fields = ["pooling_mode", "finetuning_task"]
    for field in custom_fields:
        if field in config_dict:
            print(f"Removing custom field: {field}")
            del config_dict[field]
    updated_config_path = "./fixed_config.json"
    with open(updated_config_path, 'w') as f:
        json.dump(config_dict, f, indent=2)

    print("Fixed config saved to:", updated_config_path)
    return updated_config_path, config_dict
def create_fixed_model(config_dict, original_model_path):
    config = AutoConfig.from_dict(config_dict)

    new_model = Wav2Vec2ForSequenceClassification(config)

    original_weights = torch.load(original_model_path, map_location='cpu')

    new_state_dict = {}

    for key, value in original_weights.items():
        if key.startswith("wav2vec2."):
            new_state_dict[key] = value
        elif key.startswith("classifier."):
            new_key = key.replace("classifier.", "classifier.")
            new_state_dict[new_key] = value
        else:
            new_state_dict[key] = value
    missing_keys, unexpected_keys = new_model.load_state_dict(new_state_dict, strict=False)

    if missing_keys:
        print("Missing keys:", missing_keys)
    if unexpected_keys:
        print("Unexpected keys:", unexpected_keys)
    new_model.save_pretrained("./fixed_model")
    print("Fixed model saved to ./fixed_model")

    return new_model
def create_model_card():
    """Create a proper model card for audio classification"""

    model_card = """---
language: en
license: mit
tags:
- audio-classification
- wav2vec2
- dementia-detection
- speech-classification
pipeline_tag: audio-classification
datasets:
- custom
widget:
- src: https://example.com/sample.wav
  example_title: Sample Audio
model-index:
- name: wav2vec2-large-xls-r-300m-dm32
  results:
  - task:
      name: Audio Classification
      type: audio-classification
    metrics:
    - name: Accuracy
      type: accuracy
      value: 0.85
---

# Wav2Vec2 for Dementia Detection

This model is a fine-tuned version of `facebook/wav2vec2-xls-r-300m` for dementia detection from speech audio.

## Model Description

- **Task**: Audio Classification
- **Base Model**: facebook/wav2vec2-xls-r-300m
- **Classes**:
  - 0: nodementia
  - 1: dementia
- **Sampling Rate**: 16kHz
- **Audio Length**: 32 seconds

## Usage

```python
from transformers import pipeline

# Load the pipeline
classifier = pipeline(
    "audio-classification",
    model="cogniplayapp/wav2vec2-large-xls-r-300m-dm32"
)

# Use with audio file
result = classifier("path/to/audio.wav")
print(result)
```

## Training

The model was trained on a custom dataset for dementia detection from speech patterns using:
- 22 epochs
- Learning rate: 1e-4
- Batch size: 8
- Audio length: 32 seconds

## Labels

- `LABEL_0`: nodementia
- `LABEL_1`: dementia
"""

    with open("./fixed_model/README.md", "w") as f:
        f.write(model_card)

    print("Model card created!")
def create_feature_extractor_config():
    """Create feature extractor configuration"""

    feature_config = {
        "do_normalize": True,
        "feature_extractor_type": "Wav2Vec2FeatureExtractor",
        "feature_size": 1,
        "padding_side": "right",
        "padding_value": 0.0,
        "return_attention_mask": True,
        "sampling_rate": 16000
    }

    with open("./fixed_model/preprocessor_config.json", "w") as f:
        json.dump(feature_config, f, indent=2)

    print("Feature extractor config created!")
def upload_fixed_model():
    """Upload the fixed model to Hugging Face"""

    repo_id = "cogniplayapp/wav2vec2-large-xls-r-300m-dm32"

    print("Uploading fixed model...")
    from huggingface_hub import notebook_login
    notebook_login()

    api = HfApi()
    files_to_upload = [
        "config.json",
        "pytorch_model.bin",
        "README.md",
        "preprocessor_config.json"
    ]

    for filename in files_to_upload:
        filepath = f"./fixed_model/{filename}"
        if os.path.exists(filepath):
            print(f"Uploading {filename}...")
            api.upload_file(
                path_or_fileobj=filepath,
                path_in_repo=filename,
                repo_id=repo_id,
                commit_message=f"Update {filename} for inference compatibility"
            )
        else:
            print(f"Warning: {filename} not found")

    print("Upload complete!")
def main():
    """Main function to fix your existing model"""

    print("=== Fixing Existing Hugging Face Model ===")
    config_path, model_path, preprocessor_path = download_existing_model()
    fixed_config_path, config_dict = fix_model_config(config_path)
    new_model = create_fixed_model(config_dict, model_path)
    import shutil
    shutil.copy(fixed_config_path, "./fixed_model/config.json")
    create_model_card()
    create_feature_extractor_config()
    print("\n=== Testing Fixed Model ===")
    test_model_locally()
    # upload_fixed_model()

    print("\n=== Fix Complete! ===")
    print("Your model is now ready for Hugging Face inference!")

def test_model_locally():
    """Test the fixed model locally"""
    try:
        from transformers import pipeline

        # Test loading the model
        classifier = pipeline(
            "audio-classification",
            model="./fixed_model",
            feature_extractor="./fixed_model"
        )

        print("✓ Model loaded successfully!")
        print("✓ Ready for inference!")
        # result = classifier("path/to/test/audio.wav")
        # print("Test result:", result)

    except Exception as e:
        print(f"✗ Error testing model: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

=== Fixing Existing Hugging Face Model ===


EntryNotFoundError: 404 Client Error. (Request ID: Root=1-686d9e8e-4107f7c6447953cf3346c49c;f2db1dcc-6305-43eb-ba8c-18d469879531)

Entry Not Found for url: https://huggingface.co/cogniplayapp/wav2vec2-large-xls-r-300m-dm32/resolve/main/pytorch_model.bin.